# 01 · Importación de datos
Objetivo: conectar con `hipermercado.db`, entender sus tablas y construir el **tablón analítico** (una fila por producto, tienda y día) que usaremos para el forecasting.

Recorrido: conexión → exploración → carga → calidad → `melt` de `sales` → unión de las tres tablas → guardado.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import janitor
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")

## 1. Conexión a la base de datos
`hipermercado.db` es una base **SQLite**: toda la base vive en un solo archivo.
El *engine* de SQLAlchemy es el objeto que sabe cómo conectarse a ella. La URL tiene la forma `dialecto:///ruta`.

In [ ]:
# El notebook corre desde 03_notebooks/, por eso subimos un nivel
DB_PATH = (Path("..") / "02_datos" / "01_Originales" / "hipermercado.db").resolve()
assert DB_PATH.exists(), f"No se encontró la base en {DB_PATH}"

engine = sa.create_engine(f"sqlite:///{DB_PATH}")
print("✅ Engine creado:", engine.url.database)

## 2. Qué tablas hay
El *inspector* lee la estructura de la base (tablas, columnas, claves) sin traer datos.

In [ ]:
inspector = sa.inspect(engine)
tablas = inspector.get_table_names()

with engine.connect() as conn:
    for t in tablas:
        n = conn.execute(sa.text(f'SELECT COUNT(*) FROM "{t}"')).scalar()
        print(f"{t:<12} {n:>6} filas")

## 3. Columnas de cada tabla
`sales` tiene una columna por día (`d_704`, `d_705`, ...), así que muestro esas aparte para no llenar la pantalla.

In [ ]:
for t in tablas:
    cols = inspector.get_columns(t)
    dias = [c["name"] for c in cols if c["name"].startswith("d_")]
    otras = [(c["name"], str(c["type"])) for c in cols if not c["name"].startswith("d_")]
    print(f"\n{t}")
    for nombre, tipo in otras:
        print(f"   {nombre:<14} {tipo}")
    if dias:
        print(f"   + {len(dias)} columnas de días: {dias[0]} ... {dias[-1]}")

# ¿Declara la base claves primarias o foráneas?
for t in tablas:
    print(t, "| PK:", inspector.get_pk_constraint(t)["constrained_columns"],
          "| FK:", len(inspector.get_foreign_keys(t)))

## 4. Cómo se relacionan las tablas
La base **no declara** claves primarias ni foráneas, así que las relaciones hay que deducirlas de los nombres de columna (ver `99_otros/CasoRetail_EstructuraRelaciones.jpg`):

| Tabla A | Tabla B | Columna(s) que las une |
|---|---|---|
| `sales` | `sell_prices` | `item_id` + `store_id` |
| `sell_prices` | `calendar` | `wm_yr_wk` (semana) |
| `sales` | `calendar` | `d_704`, `d_705`... en `sales` equivalen a la columna `d` de `calendar` |

`sales` está en **formato ancho** (un día por columna). Para unirla con las otras hay que pasarla a formato largo con `melt`, y eso viene en el siguiente paso.

## 5. Cargar las tablas a DataFrames
`pd.read_sql` ejecuta una consulta SQL y devuelve un DataFrame. Acá son tablas chicas, pero en bases grandes conviene traer solo lo necesario con `WHERE` o `LIMIT`.

In [ ]:
df_calendar = pd.read_sql("SELECT * FROM calendar", engine)
df_sales = pd.read_sql("SELECT * FROM sales", engine)
df_prices = pd.read_sql("SELECT * FROM sell_prices", engine)

for nombre, df in [("calendar", df_calendar), ("sales", df_sales), ("sell_prices", df_prices)]:
    print(f"{nombre:<12} {df.shape}")

### `calendar`
Traduce el código de día (`d_704`) a una fecha real y agrega contexto: día de la semana, mes, año y **eventos** (feriados, fiestas).

In [ ]:
df_calendar.drop(columns="index", inplace=True)
df_calendar.head()

### `sales`
Unidades vendidas por día. Cada fila es una **serie de tiempo** (un producto en una tienda) y cada columna `d_*` es un día. `id` identifica la serie.

Items se agregan en depts, depts se agregan en cats, cats se agregan en stores, stores perteneen a un state

In [ ]:
df_sales.drop(columns="index", inplace=True)
df_sales.iloc[:, :12].head()

### `sell_prices`
Precio de venta de cada producto, en cada tienda, por **semana** (`wm_yr_wk`). Es la tabla que nos dice cuánto costaba lo que se vendió.

In [ ]:
df_prices.drop(columns="index", inplace=True)
df_prices.head()

## 6. Chequeos básicos de calidad
Antes de transformar nada: nulos, duplicados y rango de fechas.

In [ ]:
# 'index' es una columna sobrante que dejó pandas al guardar la base
for nombre, df in [("calendar", df_calendar), ("sales", df_sales), ("sell_prices", df_prices)]:
    print(f"{nombre:<12} nulos: {int(df.isna().sum().sum()):>6} | duplicados: {int(df.duplicated().sum()):>6}")

print("\nNulos por columna en calendar:")
print(df_calendar.isna().sum()[lambda s: s > 0])

print("\nRango de fechas:", df_calendar["date"].min(), "->", df_calendar["date"].max())
print("Series en sales (id únicos):", df_sales["id"].nunique(), "| items:", df_sales["item_id"].nunique(), "| tiendas:", df_sales["store_id"].nunique())

## 7. Pasar `sales` a formato largo (`melt`)
Hoy `sales` tiene **una columna por día** (`d_704`, `d_705`, ...). Con `melt` esas columnas pasan a ser filas: queda una fila por **producto, tienda y día**, con la cantidad vendida en `ventas`.

- `id_vars`: las 6 columnas descriptivas que se repiten en cada fila.
- `var_name`: nombre de la nueva columna con el día (`d`).
- `value_name`: nombre de la nueva columna con el valor (`ventas`).

`id` se descarta porque es redundante: es `item_id` + `store_id` + el sufijo `validation`.

In [ ]:
sales_largo = df_sales.melt(
    id_vars=df_sales.columns[0:6],
    var_name="d",
    value_name="ventas",
)
sales_largo = sales_largo.drop(columns="id")

# Control: 20 series x 1095 días
dias = df_sales.shape[1] - 6
assert len(sales_largo) == len(df_sales) * dias
print(sales_largo.shape)
sales_largo.head()

## 8. Tablón analítico
Unimos las tres tablas en una sola, con **una fila por producto, tienda y día**. Usamos `left join` desde `sales_largo` para no perder ninguna venta:

1. `sales_largo` + `calendar` por `d` → agrega fecha, día de la semana, mes, año y eventos.
2. el resultado + `sell_prices` por `store_id`, `item_id` y `wm_yr_wk` → agrega el precio de esa semana.

`validate="m:1"` le dice a pandas que cada fila de la izquierda debe encontrar **como máximo una** en la derecha. Si hubiera duplicados que inflaran las filas, falla en vez de seguir en silencio.

In [ ]:
tablon = (
    sales_largo
    .merge(df_calendar, on="d", how="left", validate="m:1")
    .merge(df_prices, on=["store_id", "item_id", "wm_yr_wk"], how="left", validate="m:1")
)
tablon["date"] = pd.to_datetime(tablon["date"])

# Control: los joins no deben cambiar la cantidad de filas
assert len(tablon) == len(sales_largo)
print(tablon.shape)
tablon.head()

In [ ]:
# Nulos del tablón: ¿qué significan?
nulos = tablon.isna().sum()
print(nulos[nulos > 0])
print(f"Días sin precio: {tablon['sell_price'].isna().mean():.1%}")

Los nulos de `event_*` son días sin evento. Los de `sell_price` son semanas en que el producto **no tenía precio cargado** (típicamente, todavía no se vendía en esa tienda). Los dejamos como nulos por ahora y los tratamos en la etapa de limpieza.

In [ ]:
SALIDA = (Path("..") / "02_datos" / "03_Entrenamiento" / "tablon_analitico.parquet").resolve()
tablon.to_parquet(SALIDA, index=False)
print("✅ Tablón guardado:", SALIDA.name, tablon.shape)

## Resultado
El tablón analítico queda guardado en `02_datos/03_Entrenamiento/tablon_analitico.parquet`. **Parquet** es un formato columnar y comprimido: se lee más rápido y pesa menos que un CSV, y conserva los tipos de dato (por ejemplo, `date` sigue siendo fecha).

Siguiente paso: limpieza (nulos de `sell_price`) y análisis exploratorio.